# Score By Marker Expression

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Libraries

In [ ]:
import buckaroo  # noqa: F401
import lamindb as ln
import spatialdata as sd
import nbl
import matplotlib.pyplot as plt
from nbl.ln.featuresets import MarkerSet as ms

import numpy as np
import pandas as pd
from upath import UPath
import scanpy as sc
import seaborn.objects as so
import seaborn as sns
import sklearn.preprocessing as skpre
from itertools import combinations
from more_itertools import chunked

## Configuration

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [ ]:
from seaborn import axes_style

theme_dict = {**axes_style("whitegrid"), "grid.linestyle": ":", "axes.facecolor": "w", "axes.edgecolor": "slategray"}
# sns.set_style(**theme_dict)
so.Plot.config.theme.update(theme_dict)

In [ ]:
figures_upath = UPath("../../../../data/db/figures/scoring/")
figures_upath.mkdir(exist_ok=True, parents=True)

In [ ]:
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("DqIiYhr1OkgR0000")

In [ ]:
cluster = nbl.DaskLocalCluster(n_workers=10, threads_per_worker=2)
cluster(open_dashboard=True)

### Configure Data

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.get(description="NBL Tissue Samples", is_latest=True).path)

In [ ]:
nbl_wc_diagnosis_adata = nbl_sdata.tables["nbl_wc_diagnosis"]

In [ ]:
ratio_scores = [
    "ratio",
    "ratio_no_TH",
]
normalized_difference_scores = [
    "normalized_difference",
    "normalized_difference_no_TH",
]
log_ratio_scores = [
    "log_ratio",
    "log_ratio_no_TH",
]
scaled_difference_scores = [
    "scaled_difference",
    "scaled_difference_no_TH",
]
all_scores = ratio_scores + normalized_difference_scores + log_ratio_scores + scaled_difference_scores

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
pipeline = Pipeline(
    steps=[
        ("robust_scaler", skpre.RobustScaler(with_centering=False, with_scaling=False)),
        ("pt", skpre.QuantileTransformer(output_distribution="normal")),
        ("min_max", skpre.MinMaxScaler()),
    ]
)

In [ ]:
marker_keys = list(
    set(
        ms.ADRENERGIC.to_list()
        + ms.MESENCHYMAL.to_list()
        + ms.TISSUE_STRUCTURE.to_list()
        + ms.IMMUNE_INFILTRATE.to_list()
        + ms.CELL_SURFACE.to_list()
        + ms.STEM_CELL.to_list()
    )
)

nbl_wc_diagnosis_df_rw = (
    sc.get.obs_df(
        adata=nbl_wc_diagnosis_adata,
        keys=[
            *marker_keys,
            "log_ratio_no_TH",
        ],
    )
    .sort_values(by="log_ratio_no_TH")
    .rolling(window=10, on="log_ratio_no_TH", center=False)
    .median()
    .assign(log_ratio_no_TH_min_max_scale=lambda df: skpre.MinMaxScaler().fit_transform(X=df[["log_ratio_no_TH"]]))
    .merge(right=nbl_wc_diagnosis_adata.obs[["Risk"]], left_index=True, right_index=True)
)
nbl_wc_diagnosis_df = sc.get.obs_df(
    adata=nbl_wc_diagnosis_adata,
    keys=[
        *marker_keys,
        "log_ratio_no_TH",
        "Risk",
    ],
)

In [ ]:
nbl_wc_diagnosis_rw_melt = nbl_wc_diagnosis_df_rw.melt(
    id_vars=["log_ratio_no_TH", "log_ratio_no_TH_min_max_scale", "Risk"],
    var_name="Marker",
    value_name="Expression",
)

In [ ]:
y_max = int(np.ceil(nbl_wc_diagnosis_rw_melt["Expression"].max()))
y_min = nbl_wc_diagnosis_rw_melt["Expression"].min()

In [ ]:
len(ms.IMMUNE_INFILTRATE.to_list())

In [ ]:
fig = plt.Figure(figsize=(16, 10), dpi=300)
subfigs = fig.subfigures(nrows=1, ncols=3)

fig.suptitle(t=f"Scores v Marker Expression: {ms.MESENCHYMAL.name()}")

for idx, (sf, marker) in enumerate(zip(subfigs.flat, ms.MESENCHYMAL.to_list(), strict=True)):
    data = nbl_wc_diagnosis_rw_melt.query(f"Marker == '{marker}'")
    p = (
        so.Plot(data, x="log_ratio_no_TH_min_max_scale", y="Expression", color="Risk")
        .add(
            so.Line(),
            lowess := nbl.pl.LowessDask(frac=0.1, clip_min=0),
            legend=True,
        )
        .label(title=f"Score vs {marker}", x=r"Log Ratio Score | Min-Max Scale", y="Expression")
        .limit(
            x=(0, 1),
            y=(0, y_max),
        )
        .scale(
            x=so.Continuous().tick(at=np.linspace(0, 1, 11, endpoint=True)),
            y=so.Continuous().tick(at=np.linspace(0, y_max, y_max + 1, endpoint=True)),
        )
        .on(sf)
        .plot()
    )
    first_tick = p._figure.axes[idx].get_xticks()[0]
    last_tick = p._figure.axes[idx].get_xticks()[-1]
    p._figure.axes[idx].text(first_tick, -0.85, "ADRN", ha="center")
    p._figure.axes[idx].text(last_tick, -0.85, "MESN", ha="center")

In [ ]:
ms.MESENCHYMAL.name()

In [ ]:
fig.savefig(f"./Neuroblastoma Markers/{ms.MESENCHYMAL.name()}_risk_markers.pdf", dpi=300, bbox_inches="tight")

In [ ]:
theme_dict = {**axes_style("whitegrid"), "grid.linestyle": ":", "axes.facecolor": "w", "axes.edgecolor": "slategray"}
sns.set_style(theme_dict)

In [ ]:
intervals = pd.interval_range(start=0, end=1, periods=10)
nbl_wc_diagnosis_df = nbl_wc_diagnosis_df.assign(log_ratio_min_max=lambda df: skpre.minmax_scale(df["log_ratio_no_TH"]))
nbl_wc_diagnosis_df["log_ratio_group"] = pd.cut(nbl_wc_diagnosis_df["log_ratio_min_max"], bins=intervals)

In [ ]:
fig = plt.Figure(figsize=(16, 10), dpi=300)
# subfigs = fig.subfigures(nrows=1, ncols=1)

# fig.suptitle(t=f"Scores v Marker Expression: {ms.MESENCHYMAL.name()}")

l = ms.IMMUNE_INFILTRATE.to_list()

for idx, _immune_chunks in enumerate(chunked(l, 5)):
    p = (
        so.Plot(
            nbl_wc_diagnosis_rw_melt.query("Marker in @immune_chunks"),
            x="log_ratio_no_TH_min_max_scale",
            y="Expression",
            color="Marker",
        )
        .layout(
            size=(16, 8),
            engine="constrained",
        )
        .add(
            so.Line(),
            lowess := nbl.pl.LowessDask(frac=0.1, clip_min=0),
            legend=True,
        )
        .label(title=f"Score vs {ms.IMMUNE_INFILTRATE.name()}", x=r"Log Ratio Score | Min-Max Scale", y="Expression")
        .limit(
            x=(0, 1),
            y=(0, y_max),
        )
        .scale(
            x=so.Continuous().tick(at=np.linspace(0, 1, 11, endpoint=True)),
            y=so.Continuous().tick(at=np.linspace(0, y_max, y_max + 1, endpoint=True)),
        )
        # .on(fig)
        .plot()
    )
    p.save(f"./scores_by_{ms.IMMUNE_INFILTRATE.name()}_{idx}.pdf", dpi=300, pad_inches=1, bbox_inches="tight")

# first_tick = p._figure.axes[0].get_xticks()[0]
# last_tick = p._figure.axes[idx].get_xticks()[-1]
# p._figure.axes[idx].text(first_tick, -0.85, "ADRN", ha="center")
# p._figure.axes[idx].text(last_tick, -0.85, "MESN", ha="center")

In [ ]:
rng = np.random.default_rng(seed=0)

chunks = chunked(ms.IMMUNE_INFILTRATE.to_list(), n=5)

rng.choice(list(combinations(ms.IMMUNE_INFILTRATE.to_list(), 2)), 10)

In [ ]:
ms.TISSUE_STRUCTURE.to_list()

In [ ]:
list(combinations(["CD11B", "CD11c", "CD15", "CD3", "CD57"], 2))

In [ ]:
for marker_combo in list(combinations(["Cd14", "Cd68", "HLA Class I", "HLA Class I", "PD1", "PDL1"], 2)):
    marker_A, marker_B = marker_combo
    fig = plt.figure(figsize=(16, 2), dpi=300)
    subfigs = fig.subplots(nrows=1, ncols=10, sharex=True, sharey=True, width_ratios=[1] * 10, height_ratios=[1])

    for sf, log_ratio_group in zip(
        subfigs.flat,
        (G := nbl_wc_diagnosis_df.groupby(by="log_ratio_group", sort=True, observed=True)).groups,
        strict=False,
    ):
        data = G.get_group(log_ratio_group)
        sf.set_xlim(left=0, right=y_max)
        sf.set_ylim(bottom=0, top=y_max)
        sf.set_xticks(ticks=np.linspace(0, y_max, y_max + 1, endpoint=True))
        sf.set_yticks(ticks=np.linspace(0, y_max, y_max + 1, endpoint=True))
        sf.tick_params(axis="both", which="major", labelsize="xx-small")
        sf.set_xlabel(xlabel=marker_A, fontsize="x-small")
        sf.set_ylabel(ylabel=marker_B, fontsize="x-small")
        sf.set_title(
            label=rf"Interval: (${log_ratio_group.left:.1f}$, ${log_ratio_group.right:.1f}$), $n={len(data)}$",
            fontsize="xx-small",
        )
        sf.set_box_aspect(1)
        try:
            sns.kdeplot(
                data=data,
                x=marker_A,
                y=marker_B,
                hue="Risk",
                ax=sf,
                legend=False,
                alpha=0.5,
                warn_singular=False,
                linewidths=1,
            )
        except IndexError:
            continue
    fig_dir_path = UPath(f"./{ms.IMMUNE_INFILTRATE.name()}/")
    fig_dir_path.mkdir(exist_ok=True, parents=True)
    fig.savefig(fname=fig_dir_path / f"{marker_A}_{marker_B}_log_ratio_group.pdf", dpi=300)

fig

In [ ]:
nbl_wc_diagnosis_adata

In [ ]:
sc.tl.umap(nbl_wc_diagnosis_adata, neighbors_key="area_norm_neighbors")

In [ ]:
sc.pp.neighbors(nbl_wc_diagnosis_adata, n_neighbors=10, use_rep="area_normalized_diffmap")

In [ ]:
nbl_wc_diagnosis_adata

In [ ]:
sc.tl.draw_graph(adata=nbl_wc_diagnosis_adata, neighbors_key="area_norm_neighbors")

In [ ]:
sc.tl.paga(nbl_wc_diagnosis_adata, groups="area_norm_leiden", neighbors_key="area_norm_neighbors")

In [ ]:
rng = np.random.default_rng(seed=0)

In [ ]:
nbl_wc_diagnosis_adata[rng.choice(nbl_wc_diagnosis_adata.obs_names, size=1000), :]

In [ ]:
sc.pl.paga_compare(nbl_wc_diagnosis_adata, color="log_ratio_no_TH", projection="2d")

In [ ]:
sc.pl.paga(nbl_wc_diagnosis_adata, color="vimentin")